In [82]:
# ================================
# Titanic Survival Prediction
# End-to-End ML Project
# ================================

# 1. Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 2. Load Dataset
df = pd.read_csv('train.csv')

# 3. Basic Info (EDA)
print(df.head())
print(df.info())
print(df.describe())

# 4. Handle Missing Values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop Cabin (too many missing values)
df.drop('Cabin', axis=1, inplace=True)

# 5. Feature Engineering

# Extract Title from Name
df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

# Combine rare titles
df['Title'] = df['Title'].replace(
    ['Dr', 'Col', 'Major', 'Rev', 'Lady', 'Sir', 'Capt'],
    'Rare'
)

# Create Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create Alone feature
df['Alone'] = 0
df.loc[df['FamilySize'] == 1, 'Alone'] = 1

# Drop unnecessary columns
df.drop(['Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)

# 6. Encoding (convert categorical → numeric)
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

# 7. Define Features & Target
X = df.drop('Survived', axis=1)
y = df['Survived']

# Ensure no missing values
X = X.fillna(X.median())

# 8. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# 9. Model 1: Logistic Regression
# ================================
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
print("\nLogistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

# ================================
# 10. Model 2: Random Forest
# ================================
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("\nRandom Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

# Classification Report
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

# Train vs Test Accuracy
print("\nTrain Accuracy:", rf_model.score(X_train, y_train))
print("Test Accuracy:", rf_model.score(X_test, y_test))

# ================================
# 11. Cross Validation
# ================================
scores = cross_val_score(rf_model, X, y, cv=5)

print("\nCross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c